# Adversarial Attacks & Adversarial Training

**Goal:** Implement the FGSM attack to fool a neural network, then implement adversarial training to make it robust.

**Estimated time:** ~30 minutes

**Reference:** Goodfellow et al. 2014  [*Explaining and Harnessing Adversarial Examples*](https://arxiv.org/abs/1412.6572)

## The Idea

Imagine a neural network that reaches 99% accuracy on handwritten digit recognition. You hand it a “7” — it classifies it correctly, no problem.

Now someone adds a tiny, invisible noise pattern to that image — a few pixel values shifted by at most 0.2 out of 1.0. The same model now predicts “3” with 97% confidence, even though the image looks identical to you.

This is an **adversarial example**, and the technique to craft it is called an **adversarial attack**. Neural networks exploit statistical correlations in pixel space that humans don’t perceive — and those correlations can be deliberately exploited.

You will implement both sides of the problem:

| Concept | Role | Core idea |
|---|---|---|
| FGSM | Attack | Perturb inputs in the direction that maximises the loss |
| Adversarial training | Defence | Train on clean *and* adversarial examples |



In [ ]:
# Install required packages (Colab and local environments)
%pip install -q tensorflow numpy matplotlib

In [ ]:
import sys, os
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    import urllib.request
    HELPER_URL = 'https://raw.githubusercontent.com/eth-bmai-fs26/coding-exercises/week4/adv-attack/draft/helper.py'
    urllib.request.urlretrieve(HELPER_URL, 'helper.py')
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import helper
print(f'TensorFlow {tf.__version__}  |  Setup complete!')

## Data & Model

The cells below load MNIST, build a small multi-layer perceptron (MLP), and train it for 3 epochs.
**Run them without modifications** — this is the model you will attack in Task 1 and defend in Task 2.

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = np.expand_dims(x_train.astype('float32') / 255.0, axis=-1)
x_test  = np.expand_dims(x_test.astype('float32')  / 255.0, axis=-1)
y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test  = tf.keras.utils.to_categorical(y_test,  10)
train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train)).shuffle(10000).batch(128)
test_ds  = tf.data.Dataset.from_tensor_slices((x_test,  y_test )).batch(128)
# Fixed evaluation batch of 512 images (used in all plots below)
images_eval, labels_eval = next(iter(
    tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(512)
))
print(f'Train: {x_train.shape[0]:,} images | Test: {x_test.shape[0]:,} images')

In [ ]:
model = helper.build_model()
model.fit(train_ds, epochs=3, validation_data=test_ds, verbose=1)

## 🎯 Task 1: Implement the FGSM Attack

### The Math

FGSM (Fast Gradient Sign Method) crafts an adversarial example in a **single step**:

$$x_\text{adv} = \text{clip}\!\left(x + \varepsilon \cdot \text{sign}(\nabla_x \mathcal{L}),\; 0,\; 1\right)$$

where $\mathcal{L}$ is the cross-entropy loss and $\varepsilon$ is the perturbation strength.

The key insight: **move each pixel in the direction that increases the loss most**. The sign function normalises the gradient so that every pixel shifts by exactly $\varepsilon$, regardless of gradient magnitude.

`helper.compute_gradient(image, label, model)` already computes $\nabla_x \mathcal{L}$ for you using `tf.GradientTape`. Your job is to use it.

### What to do

Fill in the **two marked lines** inside `fgsm_attack`:
1. **Perturbation** — scale the sign of the gradient by `epsilon`
2. **Return** — add the perturbation to the image and clip to `[0.0, 1.0]`

> **Do not modify** any line without a 🎯 marker.

In [ ]:
def fgsm_attack(image, label, model, epsilon=0.2):
    """
    Apply one step of FGSM:
        x_adv = clip(x + epsilon * sign(grad_x Loss), 0, 1)
    Args:
        image:   input tensor, pixel values in [0, 1]
        label:   one-hot ground-truth label tensor
        model:   trained Keras classifier
        epsilon: perturbation strength (positive float)
    Returns:
        Adversarial image tensor, clipped to [0.0, 1.0]
    """
    gradient = helper.compute_gradient(image, label, model)
    # 🎯 Compute perturbation: epsilon * sign(gradient)
    perturbation = ___ * tf.sign(___)
    # 🎯 Add perturbation to image and clip to valid pixel range [0.0, 1.0]
    return tf.clip_by_value(image + ___, 0.0, 1.0)

In [ ]:
# Run after completing Task 1.
# Each column: original image (top) vs adversarial (bottom).
# Red title = the model was fooled.
helper.plot_adversarial_examples(model, images_eval, labels_eval, fgsm_attack, epsilon=0.2)

In [ ]:
helper.plot_accuracy_vs_epsilon(
    model, images_eval, labels_eval, fgsm_attack,
    epsilons=[0.0, 0.05, 0.1, 0.2, 0.3, 0.4],
)

### Discussion

1. At what value of $\varepsilon$ does accuracy drop below 50%? Does that surprise you, given how small the visual change looks?
2. FGSM uses `sign(gradient)` rather than the raw gradient values. What would happen if you used the raw gradient instead?
3. FGSM is a *white-box* attack — it requires access to the model’s gradients. Why would that make it harder to use against a deployed production model?



## 🎯 Task 2: Implement Adversarial Training

### The Idea

The most practical defence is to **train on adversarial examples**. At each training step:

1. Generate adversarial images from the current batch with `fgsm_attack`
2. Forward pass on **both** the clean and adversarial images
3. Average the two cross-entropy losses
4. Backpropagate and update weights as usual

This forces the model to learn features that are stable under small perturbations.

Here is the *standard* (non-robust) training step for reference:

```python
with tf.GradientTape() as tape:
    pred_clean = model(images, training=True)
    loss       = loss_fn(labels, pred_clean)
grads = tape.gradient(loss, model.trainable_variables)
optimizer.apply_gradients(zip(grads, model.trainable_variables))
```

### What to do

Fill in the **three marked lines** inside `adversarial_training`:
1. Generate adversarial images with `fgsm_attack` — pass `epsilon` from the outer scope
2. Forward pass on the adversarial images (remember `training=True`)
3. Cross-entropy loss on adversarial predictions

> **Do not modify** any line without a 🎯 marker.

In [ ]:
def adversarial_training(model, train_ds, images_eval, labels_eval, epsilon=0.2, epochs=3):
    """
    Train model on a 50/50 mix of clean and FGSM adversarial examples.
    Each step averages the clean and adversarial losses,
    then performs a standard gradient update.
    """
    optimizer = tf.keras.optimizers.Adam()
    loss_fn   = tf.keras.losses.CategoricalCrossentropy()
    for epoch in range(epochs):
        print(f'Epoch {epoch + 1}/{epochs}')
        for images, labels in train_ds:
            # 🎯 Generate adversarial images using fgsm_attack (pass epsilon)
            adv_images = fgsm_attack(images, labels, model, ___)
            with tf.GradientTape() as tape:
                pred_clean = model(images, training=True)
                # 🎯 Forward pass on the adversarial images
                pred_adv = model(___, training=True)
                loss_clean = loss_fn(labels, pred_clean)
                # 🎯 Cross-entropy loss on adversarial predictions (same labels)
                loss_adv = loss_fn(labels, ___)
                total_loss = (loss_clean + loss_adv) / 2
            grads = tape.gradient(total_loss, model.trainable_variables)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))
        helper.print_epoch_stats(
            epoch + 1, epochs, model, images_eval, labels_eval, fgsm_attack, epsilon
        )

In [ ]:
# This takes a few minutes (3 epochs, each step runs FGSM + a training update).
# Watch how adversarial accuracy improves while clean accuracy stays high.
robust_model = helper.build_model()
adversarial_training(robust_model, train_ds, images_eval, labels_eval, epsilon=0.2, epochs=3)

In [ ]:
helper.plot_robustness_comparison(
    model, robust_model, images_eval, labels_eval, fgsm_attack,
    epsilons=[0.0, 0.05, 0.1, 0.2, 0.3, 0.4],
)

In [ ]:
# Compare with the plot from Task 1: are there fewer red titles?
helper.plot_adversarial_examples(robust_model, images_eval, labels_eval, fgsm_attack, epsilon=0.2)

### Discussion

1. Look at the comparison plot. Is there a clean-accuracy cost to adversarial training? Why might that be?
2. The robust model was trained against FGSM. Would it also be robust to PGD (Projected Gradient Descent), a stronger iterative version of FGSM?
3. We averaged clean and adversarial losses with equal weight. What might happen if you used `0.7 × clean + 0.3 × adversarial`, or the opposite?


### Summary

| | Standard model | Robust model |
|---|---|---|
| Training data | Clean images only | Clean + adversarial |
| Clean accuracy | ~99% | Slightly lower |
| Adversarial accuracy at $\varepsilon = 0.2$ | Drops sharply | Much more stable |

Adversarial training is the go-to practical defence — simple to implement and effective against the attack it trained on. Research into certified robustness, randomised smoothing, and adaptive attacks pushes the frontier further.